In [6]:
# =========================================
# 🚀 IMPORT
# =========================================
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor

# =========================================
# LOAD DATA
# =========================================
df = pd.read_csv("QCFE4_Perfect_Delivery_Performance.csv")

# =========================================
# 🧠 CREATE REQUIRED FEATURES
# =========================================

# Peak Intensity (already exists OR create if needed)
def get_peak(hour):
    if 8 <= hour <= 11:
        return 2
    elif 18 <= hour <= 22:
        return 3
    else:
        return 1

df['Peak_Intensity'] = df['Hour'].apply(get_peak)

# Weekend
df['Is_Weekend'] = df['Day_of_Week'].apply(lambda x: 1 if x in ['Saturday','Sunday'] else 0)

# Festival (if not present → default 0)
if 'Is_Festival' not in df.columns:
    df['Is_Festival'] = 0

# Delivery Speed
df['Delivery_Speed_kmph'] = df['Distance_Km'] / (df['Delivery_Time_Minute'] / 60)

# =========================================
# TRAFFIC LOGIC (APPLIED HERE)
# =========================================
def traffic_level(row):
    hour = row['Hour']
    peak = row['Peak_Intensity']
    weekend = row['Is_Weekend']
    festival = row['Is_Festival']
    speed = row['Delivery_Speed_kmph']

    if (
        festival == 1 or
        ((hour in range(8, 12) or hour in range(18, 23)) and peak >= 2) or
        speed < 15
    ):
        return "High"

    elif (
        (hour in range(7, 22)) and
        (peak == 1 or weekend == 1 or speed < 25)
    ):
        return "Medium"

    else:
        return "Low"

df['Traffic_Level'] = df.apply(traffic_level, axis=1)

# =========================================
# OTHER FEATURES
# =========================================
df['Items_per_Km'] = df['Items_Count'] / (df['Distance_Km'] + 0.01)

# Clean
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

# =========================================
# ENCODING
# =========================================
le_city = LabelEncoder()
le_traffic = LabelEncoder()
le_weather = LabelEncoder()

df['City'] = le_city.fit_transform(df['City'])
df['Traffic_Level'] = le_traffic.fit_transform(df['Traffic_Level'])
df['Weather'] = le_weather.fit_transform(df['Weather'])

# =========================================
# FEATURES
# =========================================
features = [
    'Distance_Km',
    'Items_Count',
    'City',
    'Items_per_Km',
    'Traffic_Level',
    'Weather',
    'Hour'
]

target = 'Delivery_Time_Minute'

X = df[features]
y = df[target]

# =========================================
# MODEL
# =========================================
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X, y)

# =========================================
# SAVE
# =========================================
joblib.dump(model, "delivery_model.pkl")
joblib.dump(le_city, "le_city.pkl")
joblib.dump(le_traffic, "le_traffic.pkl")
joblib.dump(le_weather, "le_weather.pkl")

print("Model trained with intelligent Traffic_Level!")

Model trained with intelligent Traffic_Level!
